# DM-NeRF AMP throughput benchmark (Kaggle)

Ative GPU e Internet. Execute **Save Version → Save & Run All**. Este notebook não produz resultado científico final; mede somente desempenho/estabilidade de AMP com a configuração oficial da cena `study`.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

def run(*args):
    subprocess.run([str(x) for x in args], check=True)

run('nvidia-smi')
BASE=Path('/kaggle/temp/dissertacao-dmnerf-amp')
OUTPUT=Path('/kaggle/working/dmnerf-amp-results')
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
if BASE.exists():
    shutil.rmtree(BASE)
BASE.mkdir(parents=True,exist_ok=True)
REPO=BASE/'sdf-generator'
WORK=BASE/'work'
run('git','clone','--branch','dissertacao-2026','--single-branch',
    'https://github.com/Jfsslemos/sdf-generator.git',REPO)
run('git','-C',REPO,'rev-parse','HEAD')
run(sys.executable,REPO/'tools/dmnerf/bootstrap.py','--work',WORK)
PYTHON=WORK/'env/bin/python'
run(PYTHON,'-c','import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0), torch.__version__)')
run(PYTHON,REPO/'tools/dmnerf/prepare.py','--work',WORK)


In [ ]:
run(PYTHON,REPO/'tools/dmnerf/run.py',
    '--work',WORK,'--output',OUTPUT,'--profile','amp_pilot','--stages','train','--amp')


In [ ]:
state=json.loads((OUTPUT/'raw/study/experiment/training_state.json').read_text())
elapsed=state['elapsed_seconds']
steps=state['iteration']+1
sec_per_iter=elapsed/steps
print(json.dumps({
    'steps':steps,
    'elapsed_seconds':elapsed,
    'seconds_per_iteration':sec_per_iter,
    'iterations_per_second':1/sec_per_iter,
    'projected_hours_200k':200000*sec_per_iter/3600,
    'projected_hours_300k':300000*sec_per_iter/3600,
    'max_vram_GiB':state['max_vram_bytes']/1024**3,
},indent=2))
print('Pacote:',OUTPUT.parent/(OUTPUT.name+'-bundle.zip'))


Envie `dmnerf-amp-results-bundle.zip` ao ChatGPT. Se AMP não for suficientemente mais rápido, a próxima otimização será usar as duas T4 do Kaggle em paralelo; não inicie o treino de 200k antes dessa decisão.
